# Choose Chunk Size

Measure retrieval hit-rate for several chunk sizes instead of guessing.

**Extra requirements:** pandas

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import pandas as pd

text = load_text()
questions = load_eval_questions()

For every chunk size, rebuild the index and check whether the top-k chunks contain the evidence phrase for each question.

In [ ]:
rows = []
for size in [150, 300, 600, 1200, 2400]:
    store = build_store(chunk_text(text, size=size, overlap=size // 10))
    hits_at_3 = [hit_at_k(store.search(q["question"], 3), q["evidence"]) for q in questions]
    rows.append({"chunk_size": size, "n_chunks": len(store), "hit_rate@3": sum(hits_at_3) / len(questions)})
pd.DataFrame(rows)

Small chunks are precise but miss context; huge chunks dilute the embedding. Pick the size where the curve flattens, then confirm with generation quality.